In [1]:
import networkx as nx
import numpy as np
import pandas as pd
import os
from generate_dataset import *
import json
from sklearn.metrics import normalized_mutual_info_score
from collections import defaultdict

In [2]:
default_params = {
    'cora': {
        'k': 3,
        'positive_class_label': 3,
        'percent_positive': 0.1,
        'use_original_edges': True,
        'mst': False
    },
    'citeseer': {
        'positive_class_label': 2,
        'percent_positive': 0.1,
        'use_original_edges': True,
        'mst': False
    },
    'twitch': {
        'percent_positive': 0.1,
        'mst': False
    },
    'mnist': {
        'k': 3,
        'percent_positive': 0.1,
        'mst': False
    },
    'ionosphere': {
        'percent_positive': 0.1,
        'k': 3,
        'mst': False
    }
}

In [3]:
def assortativity_coefficient(G, label_attr='true_label'):
    """
    Compute the assortativity coefficient for a categorical node attribute.
    Measures homophily: correlation between labels of connected nodes.
    Range: [-1, 1] (positive = same labels connect more).
    """
    return nx.attribute_assortativity_coefficient(G, label_attr)

def label_consistency(G, label_attr='true_label'):
    """
    For each node, compute the fraction of neighbours with the same label.
    Returns the average consistency over all nodes.
    """
    consistencies = []
    for node in G.nodes:
        neighbors = list(G.neighbors(node))
        if not neighbors:
            continue
        same = sum(1 for nb in neighbors if G.nodes[node][label_attr] == G.nodes[nb][label_attr])
        consistencies.append(same / len(neighbors))
    return np.mean(consistencies) if consistencies else 0.0

def intra_inter_cluster_distances(G, label_attr='true_label'):
    """
    Compute average shortest path lengths for:
      - intra-cluster pairs (same label)
      - inter-cluster pairs (different labels)
    Returns a dict: {'intra_mean': ..., 'inter_mean': ..., 'ratio': ...}
    Note: Computes all-pairs shortest paths; may be heavy for large graphs.
    """
    # Get all-pairs shortest path lengths (unweighted)
    sp = dict(nx.all_pairs_shortest_path_length(G))
    nodes = list(G.nodes)
    labels = {node: G.nodes[node][label_attr] for node in nodes}

    intra_lengths = []
    inter_lengths = []
    n = len(nodes)
    for i in range(n):
        u = nodes[i]
        for j in range(i+1, n):
            v = nodes[j]
            # Check if path exists (if not, skip; you may decide to set a large value)
            if v in sp[u]:
                d = sp[u][v]
                if labels[u] == labels[v]:
                    intra_lengths.append(d)
                else:
                    inter_lengths.append(d)

    intra_mean = np.mean(intra_lengths) if intra_lengths else np.nan
    inter_mean = np.mean(inter_lengths) if inter_lengths else np.nan
    ratio = intra_mean / inter_mean if inter_mean else np.nan
    return {
        'intra_mean': intra_mean,
        'inter_mean': inter_mean,
        'ratio': ratio
    }

def davies_bouldin_index_network(G, label_attr='true_label'):
    """
    Adapted Davies–Bouldin index for network clusters.
    For each label:
        intra = average shortest path length among nodes of that label.
        For each pair of labels i,j:
            inter = average shortest path length between nodes of i and j.
    Then for each i: max_{j≠i} ( (intra_i + intra_j) / inter_ij )
    The DB index is the average of these maxima. Lower is better.
    """
    sp = dict(nx.all_pairs_shortest_path_length(G))
    nodes = list(G.nodes)
    labels = {node: G.nodes[node][label_attr] for node in nodes}
    unique_labels = list(set(labels.values()))
    label_to_nodes = defaultdict(list)
    for node, lbl in labels.items():
        label_to_nodes[lbl].append(node)

    # Compute intra-cluster average distances
    intra = {}
    for lbl in unique_labels:
        nodes_l = label_to_nodes[lbl]
        if len(nodes_l) < 2:
            intra[lbl] = 0.0  # single node cluster – no internal distance
            continue
        distances = []
        for i, u in enumerate(nodes_l):
            for v in nodes_l[i+1:]:
                if v in sp[u]:
                    distances.append(sp[u][v])
        intra[lbl] = np.mean(distances) if distances else 0.0

    # Compute inter-cluster average distances
    inter = {}
    for i, lbl1 in enumerate(unique_labels):
        for lbl2 in unique_labels[i+1:]:
            nodes1 = label_to_nodes[lbl1]
            nodes2 = label_to_nodes[lbl2]
            distances = []
            for u in nodes1:
                for v in nodes2:
                    if v in sp[u]:
                        distances.append(sp[u][v])
            avg = np.mean(distances) if distances else np.inf  # no path → infinite separation
            inter[(lbl1, lbl2)] = avg
            inter[(lbl2, lbl1)] = avg  # symmetric

    # Compute DB index
    db_values = []
    for i, lbl_i in enumerate(unique_labels):
        max_ratio = 0.0
        for lbl_j in unique_labels:
            if lbl_i == lbl_j:
                continue
            denom = inter.get((lbl_i, lbl_j), np.inf)
            if denom == 0 or np.isinf(denom):
                ratio = np.inf
            else:
                ratio = (intra[lbl_i] + intra[lbl_j]) / denom
            max_ratio = max(max_ratio, ratio)
        db_values.append(max_ratio)
    return np.mean(db_values) if db_values else np.nan

def modularity(G, label_attr='true_label'):
    """
    Compute modularity using the given node labels as the partition.
    Higher values (>0.3) suggest strong community structure.
    """
    # Build partition: list of sets of nodes per label
    label_to_nodes = defaultdict(set)
    for node, data in G.nodes(data=True):
        label_to_nodes[data[label_attr]].add(node)
    partition = list(label_to_nodes.values())
    return nx.community.modularity(G, partition)

def normalized_mutual_information(G, label_attr='true_label', community_algorithm=None):
    """
    Compare labels with a community detection result.
    By default uses Louvain (python-louvain) if available.
    Returns NMI score between the true labels and the detected communities.
    """
    # Run community detection
    if community_algorithm is None:
        try:
            from community import community_louvain
            # best_partition returns a dict: node -> community_id
            communities = community_louvain.best_partition(G, random_state=42)
        except ImportError:
            raise ImportError("Louvain algorithm not available. Install python-louvain.")
    else:
        communities = community_algorithm(G)

    # Convert communities to a uniform dictionary format
    if isinstance(communities, dict):
        comm_dict = communities
    else:
        # Assume it's a list of sets (e.g., from networkx.community.louvain_communities)
        comm_dict = {}
        for idx, comm in enumerate(communities):
            for node in comm:
                comm_dict[node] = idx

    # Build label arrays aligned by node order
    nodes = list(G.nodes)
    true_labels = [G.nodes[n][label_attr] for n in nodes]
    detected_labels = [comm_dict[n] for n in nodes]

    return normalized_mutual_info_score(true_labels, detected_labels)

def label_distribution_by_centrality(G, label_attr='true_label',
                                     centrality_func=nx.degree_centrality,
                                     bins=3, quantiles=True):
    """
    Divide nodes into centrality strata (low, medium, high) and show label distribution.
    If quantiles=True, bins are based on percentiles (equal number of nodes).
    Otherwise bins are equally spaced on the centrality range.
    Returns a pandas DataFrame with counts and proportions.
    """
    cent = centrality_func(G)
    nodes = list(G.nodes)
    centrality_vals = [cent[node] for node in nodes]
    labels = [G.nodes[node][label_attr] for node in nodes]

    df = pd.DataFrame({'node': nodes, 'centrality': centrality_vals, 'label': labels})

    # Create strata
    if quantiles:
        # Use pd.qcut for quantile-based bins
        df['stratum'] = pd.qcut(df['centrality'], q=bins, labels=[f'Q{i+1}' for i in range(bins)],
                                duplicates='drop')
    else:
        # Equal-width bins
        df['stratum'] = pd.cut(df['centrality'], bins=bins, labels=[f'Bin{i+1}' for i in range(bins)])

    # Count and proportion per stratum
    count_df = df.groupby(['stratum', 'label'], observed=False).size().unstack(fill_value=0)
    prop_df = count_df.div(count_df.sum(axis=1), axis=0)  # proportions within each stratum

    # Combine counts and proportions
    result = pd.concat([count_df.add_suffix('_count'), prop_df.add_suffix('_prop')], axis=1)
    return result


# Cora

In [4]:
params = default_params.get('cora', {}).copy()
cora = load_cora_scar(**params)
cora

Loading Cora dataset...
Graph is not connected. Extracting the largest connected component.
Cora graph loaded: 2485 nodes, 5069 edges.


In [12]:
print("Assortativity Coefficient:", assortativity_coefficient(cora))
print("Assortativity Coefficient Observed:", assortativity_coefficient(cora, 'observed_label'))

print("Label Consistency:", label_consistency(cora))
print("Label Consistency Observed:", label_consistency(cora, 'observed_label'))

print("Intra/Inter distances:", intra_inter_cluster_distances(cora))
print("Intra/Inter distances Observed:", intra_inter_cluster_distances(cora, 'observed_label'))

print("Davies-Bouldin index:", davies_bouldin_index_network(cora))
print("Davies-Bouldin index Observed:", davies_bouldin_index_network(cora, 'observed_label'))

print("Modularity:", modularity(cora))
print("Modularity Observed:", modularity(cora, 'observed_label'))

print("NMI with Louvain:", normalized_mutual_information(cora))
print("NMI with Louvain Observed:", normalized_mutual_information(cora, 'observed_label'))

print("\nLabel distribution by decoraree centrality:")
print(label_distribution_by_centrality(cora, centrality_func=nx.degree_centrality, bins=3))
print("Label distribution by decoraree centrality Observed:", label_distribution_by_centrality(cora, centrality_func=nx.degree_centrality, bins=3, label_attr='observed_label'))

Assortativity Coefficient: 0.7557846696945941
Assortativity Coefficient Observed: 0.08671280393345498
Label Consistency: 0.9057607027109917
Label Consistency Observed: 0.9535732426430146
Intra/Inter distances: {'intra_mean': np.float64(6.136552304270738), 'inter_mean': np.float64(6.5581589840207855), 'ratio': np.float64(0.935712647287553)}
Intra/Inter distances Observed: {'intra_mean': np.float64(6.291124803185021), 'inter_mean': np.float64(6.644178523737164), 'ratio': np.float64(0.9468626980309433)}
Davies-Bouldin index: 1.8494505012943607
Davies-Bouldin index Observed: 1.9014256690996405
Modularity: 0.29549370005815584
Modularity Observed: 0.004008365752069079
NMI with Louvain: 0.1566320428495581
NMI with Louvain Observed: 0.01613582507516622

Label distribution by decoraree centrality:
label    0_count  1_count    0_prop    1_prop
stratum                                      
Q1           600      299  0.667408  0.332592
Q2           634      266  0.704444  0.295556
Q3           525

# Citeseer

In [13]:
params = default_params.get('citeseer', {}).copy()
citeseer = load_citeseer_scar(**params)
citeseer

Loading CiteSeer dataset...


Graph is not connected. Extracting the largest connected component.
CiteSeer graph loaded: 2120 nodes, 3679 edges.


Processing...
Done!


In [14]:
print("Assortativity Coefficient:", assortativity_coefficient(citeseer))
print("Assortativity Coefficient Observed:", assortativity_coefficient(citeseer, 'observed_label'))

print("Label Consistency:", label_consistency(citeseer))
print("Label Consistency Observed:", label_consistency(citeseer, 'observed_label'))

print("Intra/Inter distances:", intra_inter_cluster_distances(citeseer))
print("Intra/Inter distances Observed:", intra_inter_cluster_distances(citeseer, 'observed_label'))

print("Davies-Bouldin index:", davies_bouldin_index_network(citeseer))
print("Davies-Bouldin index Observed:", davies_bouldin_index_network(citeseer, 'observed_label'))

print("Modularity:", modularity(citeseer))
print("Modularity Observed:", modularity(citeseer, 'observed_label'))

print("NMI with Louvain:", normalized_mutual_information(citeseer))
print("NMI with Louvain Observed:", normalized_mutual_information(citeseer, 'observed_label'))

print("\nLabel distribution by deciteseerree centrality:")
print(label_distribution_by_centrality(citeseer, centrality_func=nx.degree_centrality, bins=3))
print("Label distribution by deciteseerree centrality Observed:", label_distribution_by_centrality(citeseer, centrality_func=nx.degree_centrality, bins=3, label_attr='observed_label'))

Assortativity Coefficient: 0.6972894983322441
Assortativity Coefficient Observed: 0.023168210893934567
Label Consistency: 0.8689203274914987
Label Consistency Observed: 0.9533996918077035
Intra/Inter distances: {'intra_mean': np.float64(9.52365263136862), 'inter_mean': np.float64(9.008023048805894), 'ratio': np.float64(1.057241148226311)}
Intra/Inter distances Observed: {'intra_mean': np.float64(9.38444782782276), 'inter_mean': np.float64(8.26224315615558), 'ratio': np.float64(1.135823244421354)}
Davies-Bouldin index: 1.7389095497751652
Davies-Bouldin index Observed: 1.8322634853580877
Modularity: 0.31118040203941755
Modularity Observed: 0.001514993563743235
NMI with Louvain: 0.12319578638480796
NMI with Louvain Observed: 0.013625998281078126

Label distribution by deciteseerree centrality:
label    0_count  1_count    0_prop    1_prop
stratum                                      
Q1           878      228  0.793852  0.206148
Q2           268       85  0.759207  0.240793
Q3           4

### Overall Community Analysis
* **True labels**: reveal a moderately strong community structure: high assortativity, good label consistency, a modularity near 0.3, and some agreement with Louvain communities. The intra/inter distance ratio < 1 and the Davies‑Bouldin index suggest that the classes are reasonably well separated in the graph, albeit not perfectly. The centrality analysis further shows that class 0 tends to occupy hubs, while class 1 is more peripheral.

* **Observed labels**, after binarization and downsampling class 1 to 10%, completely fail to capture this structure. Assortativity drops to near zero, modularity becomes negligible, and NMI with Louvain is almost zero. The high label consistency is an artifact of class imbalance. The downsampling has effectively erased the homophily and community signal present in the true labels.

**Implication:**   
If you were to use the observed labels as ground truth for tasks like semi‑supervised learning or community evaluation, you would incorrectly conclude that the graph lacks community structure aligned with those labels. The downsampling has introduced a severe bias that masks the true organization of the network. For any meaningful community analysis, you must rely on the true labels or apply methods that correct for the downsampling (e.g., re‑weighting, using the original multi‑class labels).

# Twitch

In [15]:
params = default_params.get('twitch', {}).copy()
twitch = load_twitch_scar(**params)

print("Assortativity Coefficient:", assortativity_coefficient(twitch))
print("Assortativity Coefficient Observed:", assortativity_coefficient(twitch, 'observed_label'))

print("Label Consistency:", label_consistency(twitch))
print("Label Consistency Observed:", label_consistency(twitch, 'observed_label'))

print("Intra/Inter distances:", intra_inter_cluster_distances(twitch))
print("Intra/Inter distances Observed:", intra_inter_cluster_distances(twitch, 'observed_label'))

print("Davies-Bouldin index:", davies_bouldin_index_network(twitch))
print("Davies-Bouldin index Observed:", davies_bouldin_index_network(twitch, 'observed_label'))

print("Modularity:", modularity(twitch))
print("Modularity Observed:", modularity(twitch, 'observed_label'))

print("NMI with Louvain:", normalized_mutual_information(twitch))
print("NMI with Louvain Observed:", normalized_mutual_information(twitch, 'observed_label'))

print("\nLabel distribution by twitch centrality:")
print(label_distribution_by_centrality(twitch, centrality_func=nx.degree_centrality, bins=3))
print("Label distribution by twitch centrality Observed:", label_distribution_by_centrality(twitch, centrality_func=nx.degree_centrality, bins=3, label_attr='observed_label'))


Loading Twitch dataset from local files, filtering to PT users...
Loaded features: 168114 users
Loaded edges: 6797557 connections
Found 2536 PT users
Found 42952 connections between PT users
Graph is not connected. Handling connectivity...
Graph is not connected. Extracting the largest connected component.
Twitch-PT graph loaded: 2442 nodes, 42950 edges.
Assortativity Coefficient: -0.04110131577267059
Assortativity Coefficient Observed: 0.010823969264561915
Label Consistency: 0.5289360594744323
Label Consistency Observed: 0.921635333571599
Intra/Inter distances: {'intra_mean': np.float64(2.587671435880483), 'inter_mean': np.float64(2.6004526041526765), 'ratio': np.float64(0.9950850216413162)}
Intra/Inter distances Observed: {'intra_mean': np.float64(2.59625871024252), 'inter_mean': np.float64(2.5737759410039205), 'ratio': np.float64(1.0087353249676543)}
Davies-Bouldin index: 1.9938867943895218
Davies-Bouldin index Observed: 1.9989477449747095
Modularity: -0.019628124318148865
Modularit

### Overall Community Analysis
**True labels**: The metrics consistently show that the true node classes are not aligned with the network’s community structure. Assortativity near zero, negative modularity, intra/inter ratio ≈ 1, and NMI ≈ 0 all indicate that the labels are essentially random with respect to graph topology. The only hint of structure is a slight tendency for class 1 to be more peripheral, but this does not translate into homophily or clustering.

**Observed labels**: Downsampling class 1 to 10% produces a heavily imbalanced binary labeling that exaggerates consistency (due to majority class) but otherwise retains no community signal. Assortativity, modularity, and NMI remain near zero; the intra/inter ratio even flips to >1, suggesting that what few class 1 nodes remain are actually more distant from each other than from class 0 nodes. The centrality distribution becomes flat because the minority class is too sparse.

**Implication:**   
The Twitch-PT graph likely possesses its own community structure (as most social networks do), but the provided true labels do not correspond to those communities. Whether these labels represent some external attribute (e.g., streaming language, partner status) is unclear, but they are not homophilic. The downsampling operation further degrades any weak signal that might have existed, producing observed labels that are even less informative. Any analysis using these labels for community detection or semi‑supervised learning would be misled – the true communities must be discovered from the graph topology itself, not from these node attributes.



# MNIST

In [4]:
params = default_params.get('mnist', {}).copy()
mnist = load_mnist_scar(**params)
mnist

print("Assortativity Coefficient:", assortativity_coefficient(mnist))
print("Assortativity Coefficient Observed:", assortativity_coefficient(mnist, 'observed_label'))

print("Label Consistency:", label_consistency(mnist))
print("Label Consistency Observed:", label_consistency(mnist, 'observed_label'))

# print("Intra/Inter distances:", intra_inter_cluster_distances(mnist))
# print("Intra/Inter distances Observed:", intra_inter_cluster_distances(mnist, 'observed_label'))

# print("Davies-Bouldin index:", davies_bouldin_index_network(mnist))
# print("Davies-Bouldin index Observed:", davies_bouldin_index_network(mnist, 'observed_label'))

print("Modularity:", modularity(mnist))
print("Modularity Observed:", modularity(mnist, 'observed_label'))

print("NMI with Louvain:", normalized_mutual_information(mnist))
print("NMI with Louvain Observed:", normalized_mutual_information(mnist, 'observed_label'))

print("\nLabel distribution by MNIST centrality:")
print(label_distribution_by_centrality(mnist, centrality_func=nx.degree_centrality, bins=3))
print("Label distribution by MNIST centrality Observed:", label_distribution_by_centrality(mnist, centrality_func=nx.degree_centrality, bins=3, label_attr='observed_label'))


Loading MNIST dataset...
Graph is not connected. Extracting the largest connected component.
MNIST graph loaded: 69976 nodes, 164917 edges.
Assortativity Coefficient: 0.9228332246439802
Assortativity Coefficient Observed: 0.047036921420384334
Label Consistency: 0.95745003311027
Label Consistency Observed: 0.9113731781711755
Modularity: 0.4611224521870645
Modularity Observed: 0.004337961023626241
NMI with Louvain: 0.2523821806122547
NMI with Louvain Observed: 0.015160845170175357

Label distribution by MNIST centrality:
label    0_count  1_count    0_prop    1_prop
stratum                                      
Q1         19818    19937  0.498503  0.501497
Q2          6339     5843  0.520358  0.479642
Q3          9416     8623  0.521980  0.478020
Label distribution by MNIST centrality Observed: label    0_count  1_count    0_prop    1_prop
stratum                                      
Q1         37740     2015  0.949315  0.050685
Q2         11612      570  0.953210  0.046790
Q3         1

### Overall Community Analysis
* **True labels**: The MNIST graph, when built from the original digit classes, exhibits exceptionally strong community structure. The assortativity (0.923) and modularity (0.461) are among the highest possible, indicating that the graph is nearly perfectly partitioned by digit class. The high label consistency confirms that nodes of the same digit are tightly clustered. The NMI with Louvain (0.252) shows that the topological communities detected by Louvain align reasonably well with the true digits, though not perfectly – likely because Louvain may merge or split some digits based on local density. The centrality analysis further shows that class 0 (perhaps digit 0) tends to be slightly more central, forming a hub‑like core, while other digits are more peripheral.

* **Observed labels**: Binarizing the labels and downsampling class 1 to 10% destroys virtually all community signal. Assortativity drops to near zero, modularity becomes negligible, and NMI with Louvain is almost zero. The high observed label consistency (0.911) is an artifact of the extreme class imbalance (class 0 now ≈95% of nodes), not an indicator of structure. The centrality distribution becomes flat, revealing no relationship between class and degree.

**Implication**:
The true labels capture a very clear community structure that is deeply embedded in the graph topology. The downsampling operation, intended perhaps to simulate a scarce labeled scenario, has erased that structure. If one were to use the observed labels for tasks like semi‑supervised node classification or community evaluation, they would mistakenly conclude that the graph lacks meaningful communities aligned with those labels. The correct conclusion is that the graph is highly structured, but the observed labels are too noisy and imbalanced to reflect it. Any analysis should rely on the true multi‑class labels or apply methods that account for the downsampling bias.



# Ionosphere

In [5]:
params = default_params.get('ionosphere', {}).copy()
ionosphere = load_ionosphere_scar(**params)
ionosphere

print("Assortativity Coefficient:", assortativity_coefficient(ionosphere))
print("Assortativity Coefficient Observed:", assortativity_coefficient(ionosphere, 'observed_label'))

print("Label Consistency:", label_consistency(ionosphere))
print("Label Consistency Observed:", label_consistency(ionosphere, 'observed_label'))

print("Intra/Inter distances:", intra_inter_cluster_distances(ionosphere))
print("Intra/Inter distances Observed:", intra_inter_cluster_distances(ionosphere, 'observed_label'))

print("Davies-Bouldin index:", davies_bouldin_index_network(ionosphere))
print("Davies-Bouldin index Observed:", davies_bouldin_index_network(ionosphere, 'observed_label'))

print("Modularity:", modularity(ionosphere))
print("Modularity Observed:", modularity(ionosphere, 'observed_label'))

print("NMI with Louvain:", normalized_mutual_information(ionosphere))
print("NMI with Louvain Observed:", normalized_mutual_information(ionosphere, 'observed_label'))

print("\nLabel distribution by ionosphere centrality:")
print(label_distribution_by_centrality(ionosphere, centrality_func=nx.degree_centrality, bins=3))
print("Label distribution by ionosphere centrality Observed:", label_distribution_by_centrality(ionosphere, centrality_func=nx.degree_centrality, bins=3, label_attr='observed_label'))


Loading Ionosphere dataset...
Ionosphere graph loaded: 351 nodes, 896 edges.
Assortativity Coefficient: 0.5943086340123618
Assortativity Coefficient Observed: 0.06842237835694168
Label Consistency: 0.8070411070411071
Label Consistency Observed: 0.8883281327725773
Intra/Inter distances: {'intra_mean': np.float64(6.080786092214663), 'inter_mean': np.float64(6.082998236331569), 'ratio': np.float64(0.9996363398391777)}
Intra/Inter distances Observed: {'intra_mean': np.float64(6.042279513536457), 'inter_mean': np.float64(6.377728654324399), 'ratio': np.float64(0.9474030397074839)}
Davies-Bouldin index: 1.8252838454535438
Davies-Bouldin index Observed: 1.9970191580517254
Modularity: 0.2599593181999363
Modularity Observed: 0.007951386120854628
NMI with Louvain: 0.15443508549787294
NMI with Louvain Observed: 0.022533909507710656

Label distribution by ionosphere centrality:
label    0_count  1_count    0_prop    1_prop
stratum                                      
Q1            89       89  0.

### Overall Community Analysis
* **True labels**: The ionosphere graph exhibits moderate community structure aligned with the true classes. Assortativity (0.594), modularity (0.260), and label consistency (0.807) all point to meaningful homophily. However, the distance‑based measures (intra/inter ratio ≈ 1, DB index 1.825) suggest that clusters are not strongly separated in terms of shortest paths – possibly because the graph is dense or path lengths are short overall. The NMI with Louvain (0.154) indicates that the topological communities detected by Louvain only partially match the true labels. The centrality analysis reveals a clear pattern: class 1 nodes tend to be high‑degree hubs, forming a core, while class 0 nodes are more peripheral. This degree‑label correlation likely drives the observed homophily (hubs connect to hubs).

* **Observed labels**: Binarizing and downsampling class 1 to 10% erases nearly all community signal. Assortativity, modularity, and NMI drop to near zero. The intra/inter ratio becomes slightly <1, but the DB index worsens to near 2, indicating poor separation. The high observed label consistency (0.888) is an artifact of class imbalance. The centrality distribution becomes flat, masking the relationship between class and degree.

**Implication**:
The true labels capture a non‑trivial community structure, with class 1 occupying a central, hub‑like role. This structure, while not as strong as in the MNIST graph, is still clearly present. The downsampling operation destroys this signal, producing observed labels that are essentially random with respect to the graph’s topology. Any analysis using the observed labels would severely underestimate the community structure inherent in the true classes. For accurate community or semi‑supervised learning, one must rely on the true multi‑class labels or correct for the downsampling bias.